# Agentic Email - Property Highlight Generator

This notebook takes input data table that has user-property matched. Based on the property details provided in each line, and user input writing instructions, generate highlight for each property.

In [ ]:
# Install dependencies (run once)
%pip install openai --quiet

In [9]:
import pandas as pd
import os
from dotenv import load_dotenv

## Step 1 — Azure OpenAI Credentials

Fill in your Azure OpenAI endpoint, API key, and deployment name below.
These match the values in the dashboard's `.env.local` file.

In [ ]:
load_dotenv()
AZURE_OPENAI_ENDPOINT    = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY     = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_DEPLOYMENT  = os.getenv("AZURE_OPENAI_DEPLOYMENT")

## Step 2 — Inputs

Three inputs are required. Edit these to test different scenarios.

### 2a. User-Property Matched Table
This will be taken from the Audience-Property Agent (Genie Space) output

In [14]:
# Some question example hard copies

questions = [
    """Retrieve customer that saved at least 2 properties in the past 7 days. 
    Find the last 2 properties they saved and last 2 properties they viewed."""

    , """Retrieve the last 4 saved or viewed properties in the past 7 days 
    for each customer. Deduplicate between saved and viewed list giving saved 
    higher priority. If there are overlaps, keep properties in saved and 
    continue searching for viewed to fill the list. Label each property with 
    their action type."""

    , """Retrieve customers that registered for any auction events in the past 7 days. 
    Get top 2 properties for those events."""

    , """Get 4 least engaged (save or view) properties for the upcoming auction events."""

]

In [18]:
def build_query_prompt(question: str) -> str:
    return f"""
    {question}
    Always include these property detail columns: ListingPrice, StreetNumber,
    StreetName, City, State, County, TransactionType, SQFT. Drop properties that
    do not have all the details.
    Include other columns relevant to the question.
    Drop customers that do not meet the property count threshold.
    """

In [19]:
query_prompt = build_query_prompt(questions[0])
print(query_prompt)


    Retrieve customer that saved at least 2 properties in the past 7 days. 
    Find the last 2 properties they saved and last 2 properties they viewed.
    Always include these property detail columns: ListingPrice, StreetNumber,
    StreetName, City, State, County, TransactionType, SQFT. Drop properties that
    do not have all the details.
    Include other columns relevant to the question.
    Drop customers that do not meet the property count threshold.
    


In [20]:
from genie_audience_property_query import genie_audience_property_query

result = genie_audience_property_query(query_prompt)

query       = result["query"]
df          = result["result_table"]
result_json = result["result_json"]
conversation_id = result["conversation_id"]


Status: FILTERING_CONTEXT
Status: ASKING_AI
Status: ASKING_AI
Status: PENDING_WAREHOUSE
Status: PENDING_WAREHOUSE
Status: ASKING_AI
Status: COMPLETED


### 2a'. Deduplicate the property list 
So only one highlight is written for each unique property

In [21]:
exclude_cols = ["xome_user_id", "event_type", "event_datetime"]
unique_properties = (df
                     .drop(columns=[c for c in exclude_cols if c in df.columns])
                     .drop_duplicates(subset=["listing_id"])
)
unique_properties.head(10)


,property_action,listing_id,ListingPrice,StreetNumber,StreetName,City,State,County,TransactionType,SQFT
0,save,405286049,0,664,Friesland,Hampton,GA,HENRY,FORECLOSURE_TRUSTEE,3058
1,save,387408063,233740,1104,Laflor,Mcdonough,GA,HENRY,FORECLOSURE_TRUSTEE,2412
2,view,416500328,0,568,CALEDON,Hampton,GA,CLAYTON,FORECLOSURE_TRUSTEE,3198
3,view,416908994,0,1305,DEUTZ,Locust Grove,GA,HENRY,FORECLOSURE_TRUSTEE,2619
4,save,417529726,150000,4,BROAD,Binghamton,NY,BROOME,NEWLY_FORECLOSED,1996
5,save,418331794,315000,351,QUINCY,Bronx,NY,BRONX,REO,1400
6,view,411033925,67500,29,Julian,Binghamton,NY,BROOME,NEWLY_FORECLOSED,1063
8,save,417222772,285000,8210,BIRDSONG,Fort Washington,MD,PRINCE GEORGES,REO,2101
9,save,401648279,0,7004,Sand Cherry,Clinton,MD,PRINCE GEORGE'S,FORECLOSURE_TRUSTEE,3264
10,view,411855317,420000,13902,Gold Bottom,Brandywine,MD,PRINCE GEORGE'S,NEWLY_FORECLOSED,3564


### 2b. Writing Instructions

Writing insturctions for the property highlights.
Using hard copied values here. In the real app will allow user input.

In [23]:
instructions = [
    "follow this format: $ListingPrice | City, State | TransactionType | Beds | Baths | SQFT"
    , "with captivating messaging encouraging customer engagements. Do not exceed 50 words"
    , "with captivating messaging encouraging customers to participate in the auction. Do not exceed 50 words"
]

In [ ]:

def get_writing_instruction() -> str:
    """Prompt the user to enter a writing instruction for the highlights."""
    instruction = input("Enter writing instruction for property highlights:\n> ").strip()
    return instruction


## Step 3 — Generator Logic

Helper functions and the prompt builder. No edits needed here.

In [10]:
# This function takes one row in a dataframe at a time
def format_property_for_prompt(row: pd.Series) -> str:
    ID_FIELDS = {
        "id", "xome_user_id", "user_id", "customer_id",
        "listing_id", "listingid", "asset_id", "property_id",
    }

    """Auto-format a single dataframe row, excluding ID fields."""
    lines = []
    for key, value in row.items():
        if key.lower().replace("_", "").replace("-", "") in {f.replace("_", "") for f in ID_FIELDS}:
            continue
        if value is None or value == "" or pd.isna(value):
            continue
        label = key.replace("_", " ").title()
        lines.append(f"  {label}: {value}")
    return "- Property:\n" + "\n".join(lines)


In [ ]:

# Build one prompt for each row in the property dataframe
def build_prompt(row: pd.Series, instruction: str) -> str:
    property_block = format_property_for_prompt(row)

    return f"""You are a real estate copywriter.

    PROPERTY:
    {property_block}

    Write highlight for this property {instruction}
    If any property attributes are 0 or not available, skip that attribute completely, 
    do not use the 0 or null values in the writing.
    """


## Step 4 — Run the Generator

In [ ]:
import re
from openai import AzureOpenAI

client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2025-01-01-preview",
)

instruction = instructions[0]

results = []

for _, row in unique_properties.iterrows():
    prompt = build_prompt(row, instruction)
    
    response = client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        messages=[
            {"role": "system", "content": "You are a real estate copywriter. Write concise, compelling property highlights."},
            {"role": "user", "content": prompt},
        ],
        max_tokens=500,
        temperature=0.7,
    )
    
    highlight = response.choices[0].message.content.strip()
    results.append({**row.to_dict(), "highlight": highlight})

df_highlights = pd.DataFrame(results)
# df_highlights.head()


,property_action,listing_id,ListingPrice,StreetNumber,StreetName,City,State,County,TransactionType,SQFT,highlight
0,save,405286049,0,664,Friesland,Hampton,GA,HENRY,FORECLOSURE_TRUSTEE,3058,"$0 | Hampton, GA | Foreclosure Trustee | Spaci..."
1,save,387408063,233740,1104,Laflor,Mcdonough,GA,HENRY,FORECLOSURE_TRUSTEE,2412,"$233,740 | McDonough, GA | Foreclosure Trustee..."
2,view,416500328,0,568,CALEDON,Hampton,GA,CLAYTON,FORECLOSURE_TRUSTEE,3198,"$0 | Hampton, GA | Foreclosure Trustee | 0 Bed..."
3,view,416908994,0,1305,DEUTZ,Locust Grove,GA,HENRY,FORECLOSURE_TRUSTEE,2619,"$0 | Locust Grove, GA | Foreclosure Trustee | ..."
4,save,417529726,150000,4,BROAD,Binghamton,NY,BROOME,NEWLY_FORECLOSED,1996,"$150,000 | Binghamton, NY | Newly Foreclosed |..."
5,save,418331794,315000,351,QUINCY,Bronx,NY,BRONX,REO,1400,"$315,000 | Bronx, NY | REO | 3 Beds | 2 Baths ..."
6,view,411033925,67500,29,Julian,Binghamton,NY,BROOME,NEWLY_FORECLOSED,1063,"$67,500 | Binghamton, NY | Newly Foreclosed | ..."
7,save,417222772,285000,8210,BIRDSONG,Fort Washington,MD,PRINCE GEORGES,REO,2101,"$285,000 | Fort Washington, MD | REO | 4 Beds ..."
8,save,401648279,0,7004,Sand Cherry,Clinton,MD,PRINCE GEORGE'S,FORECLOSURE_TRUSTEE,3264,"$0 | Clinton, MD | Foreclosure Trustee | 4 Bed..."
9,view,411855317,420000,13902,Gold Bottom,Brandywine,MD,PRINCE GEORGE'S,NEWLY_FORECLOSED,3564,"$420,000 | Brandywine, MD | Newly Foreclosed |..."


## Step 5 — Output

In [28]:
df_mapped = df.merge(
    df_highlights[["listing_id", "highlight"]],
    on="listing_id",
    how="left"
)
json_mapped = df_mapped.to_dict(orient="records")
json_mapped

# df_mapped.head()

[{'property_action': 'save',
  'xome_user_id': '2259742',
  'listing_id': '405286049',
  'event_datetime': '2026-03-28 02:42:58.064441 UTC',
  'ListingPrice': '0',
  'StreetNumber': '664',
  'StreetName': 'Friesland',
  'City': 'Hampton',
  'State': 'GA',
  'County': 'HENRY',
  'TransactionType': 'FORECLOSURE_TRUSTEE',
  'SQFT': '3058',
  'highlight': '$0 | Hampton, GA | Foreclosure Trustee | Spacious 3058 SQFT | Ideal opportunity for investors!'},
 {'property_action': 'save',
  'xome_user_id': '2259742',
  'listing_id': '387408063',
  'event_datetime': '2026-03-28 02:42:43.730837 UTC',
  'ListingPrice': '233740',
  'StreetNumber': '1104',
  'StreetName': 'Laflor',
  'City': 'Mcdonough',
  'State': 'GA',
  'County': 'HENRY',
  'TransactionType': 'FORECLOSURE_TRUSTEE',
  'SQFT': '2412',
  'highlight': '$233,740 | McDonough, GA | Foreclosure Trustee | Spacious 4 Beds | 3 Baths | 2,412 SQFT'},
 {'property_action': 'view',
  'xome_user_id': '2259742',
  'listing_id': '416500328',
  'event_